# Модуль 1 · Алгоритмическая база
**Практикум.** Как найти дубли среди 100 000 снимков, не сравнивая каждую пару? Как утечка между обучением и
проверкой делает метрику красивой и ложной? Что теряет модель, если сжать её веса?

**Как работать.** Меню **«Среда выполнения → Выполнить все»** (Runtime → Run all) — и читайте сверху вниз.
В ячейках с полями справа меняйте значения и запускайте ячейку заново (▶ слева или Ctrl+Enter). Сломать ничего
нельзя: **«Среда выполнения → Отключиться и удалить среду»** вернёт всё как было. Код можно не читать — но если
интересно, он открыт и короткий.

Снимки листьев — открытый набор **PlantVillage** (Mohanty, Hughes, Salathé, 2016), лицензия CC BY-SA 3.0, репозиторий [spMohanty/PlantVillage-Dataset](https://github.com/spMohanty/PlantVillage-Dataset).

In [ ]:
#@title Подготовка: библиотеки и доступ к снимкам PlantVillage
import io, os, subprocess, urllib.parse, urllib.request, warnings
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
warnings.filterwarnings('ignore')

BASE = 'https://raw.githubusercontent.com/spMohanty/PlantVillage-Dataset/master/'
SAMPLES = {
    'apple_rust': 'raw/color/Apple___Cedar_apple_rust/46597071-9826-4f76-b853-083b76c5ef63___FREC_C.Rust 3666.JPG',
    'corn_gls': 'raw/color/Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot/8858dd29-eac0-443a-8213-425bbe223f7e___RS_GLSp 7334.JPG',
    'corn_healthy': 'raw/color/Corn_(maize)___healthy/3ac3f13d-352c-4390-bf06-94b5334cbfd8___R.S_HL 8206 copy 2.jpg',
    'grape_brot': 'raw/color/Grape___Black_rot/83b866ce-0785-4312-8e43-1ae8aa59d3bd___FAM_B.Rot 3518.JPG',
    'peach_bact': 'raw/color/Peach___Bacterial_spot/1df78012-4b87-40f4-bfa1-b868465c0b4d___Rut._Bact.S 1263.JPG',
    'soy_healthy': 'raw/color/Soybean___healthy/8f1ff646-db74-4924-abe8-373894711d26___RS_HL 5701.JPG',
    'tom_bact': 'raw/color/Tomato___Bacterial_spot/47798da0-ff59-497e-a2bb-e169c9138a7e___GCREC_Bact.Sp 3120.JPG',
    'tom_healthy': 'raw/color/Tomato___healthy/18fa8f7f-ef94-4a3b-9c79-5f4362323134___RS_HL 9977.JPG',
    'tom_late': 'raw/color/Tomato___Late_blight/6578c88a-fd14-450b-8e75-c94571decf83___GHLB2 Leaf 8849.JPG',
    'tom_mites': 'raw/color/Tomato___Spider_mites Two-spotted_spider_mite/2c2f7f98-8c5b-4981-bb74-f827c5f963fe___Com.G_SpM_FL 8734.JPG',
    'tom_mosaic': 'raw/color/Tomato___Tomato_mosaic_virus/6ebb483a-d1b6-4d7d-aad1-e717c2a36167___PSU_CG 2286.JPG',
    'tom_septoria': 'raw/color/Tomato___Septoria_leaf_spot/9acdb655-bca7-4223-a8e2-c201f6b3d188___Matt.S_CG 6834.JPG',
    'tom_target': 'raw/color/Tomato___Target_Spot/63a17b2f-817a-4717-9964-76ee7d6c3a63___Com.G_TgS_FL 9938.JPG',
    'tom_ylcv': 'raw/color/Tomato___Tomato_Yellow_Leaf_Curl_Virus/39b9e3b8-d3ba-4c0f-92be-e75ba728bce8___YLCV_GCREC 2711.JPG',
}

def get_image(path):
    """Скачать один снимок набора по его пути."""
    with urllib.request.urlopen(BASE + urllib.parse.quote(path), timeout=60) as r:
        return np.asarray(Image.open(io.BytesIO(r.read())).convert('RGB'))

def get_many(paths):
    """Скачать много снимков параллельно."""
    with ThreadPoolExecutor(16) as ex:
        return list(ex.map(get_image, paths))

def sample(name):
    return get_image(SAMPLES[name])

def list_dataset():
    """Имена всех снимков набора — без скачивания самих файлов (секунд 20 при первом вызове)."""
    if not os.path.exists('pv_list.txt'):
        if not os.path.exists('pv_repo'):
            subprocess.run(['git', 'clone', '-q', '--filter=blob:none', '--no-checkout', '--depth', '1',
                            'https://github.com/spMohanty/PlantVillage-Dataset', 'pv_repo'], check=True)
        out = subprocess.run(['git', '-C', 'pv_repo', 'ls-tree', '-r', '--name-only', 'HEAD', 'raw/color'],
                             capture_output=True, text=True, check=True).stdout
        with open('pv_list.txt', 'w') as f:
            f.write(out)
    names = open('pv_list.txt').read().splitlines()
    return [p for p in names if p.lower().endswith(('.jpg', '.jpeg', '.png'))]

def cls(path):
    return path.split('/')[2]

def show(images, titles=None, cols=6, size=2.3, colors=None):
    rows = (len(images) + cols - 1) // cols
    fig, axs = plt.subplots(rows, cols, figsize=(cols * size, rows * (size + 0.5)), squeeze=False)
    for ax in axs.ravel():
        ax.axis('off')
    for i, im in enumerate(images):
        ax = axs.ravel()[i]
        ax.imshow(im, cmap='gray' if im.ndim == 2 else None)
        if titles:
            ax.set_title(titles[i], fontsize=9, color=colors[i] if colors else 'black')
    plt.tight_layout(); plt.show()

print('Готово. Снимков-образцов:', len(SAMPLES))

## 1. Сколько стоит алгоритм: замер
Миллион чисел: отсортировать всё или только найти десять самых больших? Найти элемент в списке или в
хеш-таблице? Время зависит от компьютера, поэтому смотрим на «во сколько раз».

In [ ]:
#@title Замер: сортировка против топ-K, список против хеш-таблицы
import time
n = 1000000  #@param {type:"integer"}
rng = np.random.default_rng(0)
arr = rng.random(n)

def best_time(f, rep=3):
    t_best = 1e9
    for _ in range(rep):
        t = time.perf_counter(); f(); t_best = min(t_best, time.perf_counter() - t)
    return t_best

t_sort = best_time(lambda: np.sort(arr))
t_top = best_time(lambda: np.argpartition(arr, -10)[-10:])
lst = list(arr[:200_000]); st = set(lst); probe = list(rng.random(200))
t_list = best_time(lambda: [p in lst for p in probe], rep=1) / 200
t_set = best_time(lambda: [p in st for p in probe]) / 200
print(f'Отсортировать {n:,} чисел: {t_sort * 1000:.1f} мс; найти топ-10: {t_top * 1000:.1f} мс '
      f'→ в {t_sort / t_top:.1f} раза быстрее')
print(f'Найти элемент среди 200 000: список {t_list * 1e6:.0f} мкс, хеш-таблица {t_set * 1e6:.2f} мкс '
      f'→ в {t_list / t_set:,.0f} раз быстрее')

## 2. Два вида отпечатков: sha256 и pHash
Криптографический хеш (sha256) меняется от любого изменения файла. Перцептивный хеш (pHash) описывает крупную
форму снимка и почти не меняется, если снимок уменьшить или пересжать. Расстояние между pHash — число
несовпавших битов из 64.

In [ ]:
#@title Один лист в шести вариантах
import hashlib

from scipy.fft import dctn
from skimage import color, transform

def phash(img, n=8):
    """Перцептивный хеш: 64 бита о «крупной форме» снимка."""
    g = transform.resize(color.rgb2gray(img), (32, 32), anti_aliasing=True)
    d = dctn(g, norm='ortho')[:n, :n]
    return (d > np.median(d)).ravel()

def jpeg(img, q):
    b = io.BytesIO(); Image.fromarray(img).save(b, 'JPEG', quality=q); return np.asarray(Image.open(b).convert('RGB'))

src = sample('apple_rust'); h, w = src.shape[:2]
variants = [('Оригинал', src),
            ('Уменьшен вдвое', np.asarray(Image.fromarray(src).resize((w // 2, h // 2)))),
            ('JPEG 20 %', jpeg(src, 20)),
            ('Обрезан на 8 %', src[10:h - 10, 10:w - 10]),
            ('Повёрнут на 3°', (transform.rotate(src, 3, mode='edge') * 255).astype(np.uint8)),
            ('Другой лист', sample('grape_brot'))]
p0 = phash(src)
titles = []
for t, im in variants:
    sha = hashlib.sha256(im.tobytes()).hexdigest()[:8]
    titles.append(f'{t}\nsha256 {sha}…\npHash: {int((phash(im) != p0).sum())} из 64')
show([im for _, im in variants], titles, cols=6, size=2.6)

**Что заметить.** sha256 разный у всех шести — точный хеш ловит только точные копии. pHash у копий отличается на
0–4 бита из 64, у другого листа — на 26.

## 3. Задание 1. Почти-дубли среди 100 снимков
Берём 88 случайных снимков набора и подмешиваем 12 изменённых копий некоторых из них (уменьшение, сжатие,
обрезка, яркость, поворот, зеркальное отражение). Найдите пары с расстоянием pHash не больше порога и проверьте
глазами, где порог ошибается.

In [ ]:
#@title Собираем 100 снимков (около минуты)
ALL = list_dataset()
rng = np.random.default_rng(42)
picked = [ALL[i] for i in rng.choice(len(ALL), 88, replace=False)]
imgs = get_many(picked)
labels = [cls(p) for p in picked]
ops = [('уменьшен', lambda im: np.asarray(Image.fromarray(im).resize((128, 128)))),
       ('JPEG 30 %', lambda im: jpeg(im, 30)),
       ('обрезан 6 %', lambda im: im[8:-8, 8:-8]),
       ('ярче', lambda im: np.clip(im.astype(float) * 1.15, 0, 255).astype(np.uint8)),
       ('повёрнут 2°', lambda im: (transform.rotate(im, 2, mode='edge') * 255).astype(np.uint8)),
       ('отражён', lambda im: im[:, ::-1])]
planted = {}
for k in range(12):
    name, f = ops[k % 6]
    planted[len(imgs)] = (k * 7, name)          # копия снимка №k*7
    imgs.append(f(imgs[k * 7])); labels.append(labels[k * 7])
H = np.array([phash(im) for im in imgs])
print('Снимков:', len(imgs), '· из них подмешанных копий:', len(planted))

In [ ]:
#@title Ищем почти-дубли: все пары
threshold = 8  #@param {type:"slider", min:0, max:20, step:1}
n = len(imgs)
D = (H[:, None, :] != H[None, :, :]).sum(-1)
pairs = sorted([(i, j, D[i, j]) for i in range(n) for j in range(i + 1, n) if D[i, j] <= threshold], key=lambda p: p[2])
found_planted = {j for i, j, _ in pairs if j in planted and planted[j][0] == i}
print(f'Сравнили {n * (n - 1) // 2:,} пар. Нашли пар с расстоянием ≤ {threshold}: {len(pairs)}')
print(f'Из 12 подмешанных копий нашли {len(found_planted)}')
miss = [(j, planted[j]) for j in planted if j not in found_planted]
for j, (i, how) in miss:
    print(f'  пропущена: копия №{j} («{how}») снимка №{i}, расстояние {D[i, j]}')
show_imgs, show_t, show_c = [], [], []
for i, j, d in pairs[:12]:
    mark = 'копия: ' + planted[j][1] if j in planted and planted[j][0] == i else 'в самом наборе'
    show_imgs += [imgs[i], imgs[j]]
    show_t += [f'№{i}', f'№{j} · {d} бит\n{mark}']
    show_c += ['black', 'green' if 'копия' in mark else 'red']
if show_imgs:
    show(show_imgs, show_t, cols=6, size=2.2, colors=show_c)

**Что заметить и что сделать.**
1. Какую копию порог пропустил? Почему pHash не узнаёт зеркальный снимок, хотя человек узнаёт сразу?
2. Найдены ли пары **в самом наборе** (красная подпись)? Посмотрите на них: это дубли или просто похожие листья?
3. Подвигайте порог: при каком значении начинаются ложные пары, при каком теряются настоящие?

## 4. Как не сравнивать каждую пару
Для 100 снимков пар 4 950, для 100 000 — уже 5 миллиардов. Трюк: режем 64 бита pHash на 8 кусков по 8 бит и
раскладываем снимки по хеш-таблицам — по одной на кусок. Сравниваем только те пары, что хоть раз попали в одну
ячейку. Если две копии отличаются не больше чем на 7 бит, хотя бы один кусок у них совпадёт целиком — значит,
такую пару трюк не пропустит.

In [ ]:
#@title Кандидаты через хеш-таблицы
from collections import defaultdict
bands = 8  #@param {type:"slider", min:2, max:16, step:1}
width = 64 // bands
cand = set()
for b in range(bands):
    table = defaultdict(list)
    for i in range(n):
        table[H[i, b * width:(b + 1) * width].tobytes()].append(i)
    for ids in table.values():
        cand.update((ids[a], ids[c]) for a in range(len(ids)) for c in range(a + 1, len(ids)))
true_pairs = {(i, j) for i, j, _ in pairs}
print(f'Кусков: {bands} по {width} бит. Пар-кандидатов: {len(cand):,} из {n * (n - 1) // 2:,} '
      f'({len(cand) / (n * (n - 1) / 2):.1%})')
print(f'Из {len(true_pairs)} пар с расстоянием ≤ {threshold} среди кандидатов: {len(true_pairs & cand)}')

**Задание.** Сделайте кусков меньше (2–4) и больше (12–16). Как меняется число кандидатов и сколько настоящих пар
теряется? Это и есть выбор между скоростью и полнотой.

## 5. Задание 2. Утечка своими руками
Каждый объект снят серией из 4 почти одинаковых кадров (как соседние кадры одного вылета). Если делить данные
**по кадрам**, соседние кадры попадают и в обучение, и в проверку — модель «узнаёт» их, а не распознаёт.

In [ ]:
#@title Случайное деление против деления по объектам
frames = 4  #@param {type:"slider", min:1, max:8, step:1}
jitter = 0.3  #@param {type:"slider", min:0.0, max:2.0, step:0.1}
from scipy import ndimage as ndi
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.neighbors import KNeighborsClassifier
Xd0, yd0 = load_digits(return_X_y=True); im8 = Xd0.reshape(-1, 8, 8)
rng = np.random.default_rng(1)
Xs, ys, groups = [], [], []
for g, i in enumerate(rng.choice(len(Xd0), 600, replace=False)):
    for k in range(frames):
        im = ndi.shift(im8[i], rng.uniform(-jitter, jitter, 2), order=1) + rng.normal(0, jitter, (8, 8))
        Xs.append(np.clip(im, 0, 16).ravel()); ys.append(yd0[i]); groups.append(g)
Xs, ys, groups = np.array(Xs), np.array(ys), np.array(groups)
Xa, Xb, ya, yb = train_test_split(Xs, ys, test_size=0.3, random_state=1)
err_rand = 1 - (KNeighborsClassifier(1).fit(Xa, ya).predict(Xb) == yb).mean()
tr, te = next(GroupShuffleSplit(1, test_size=0.3, random_state=1).split(Xs, ys, groups))
err_grp = 1 - (KNeighborsClassifier(1).fit(Xs[tr], ys[tr]).predict(Xs[te]) == ys[te]).mean()
print(f'600 объектов × {frames} кадров = {len(Xs)} изображений')
print(f'Ошибка при случайном делении по кадрам: {err_rand:.1%}')
print(f'Ошибка при делении по объектам:         {err_grp:.1%}   ← это и есть честная оценка')

**Что заметить.** При случайном делении ошибка в несколько раз меньше — но это обман: модель видела «соседа» каждого
проверочного кадра. Поставьте `frames = 1` — разница исчезнет, потому что соседей больше нет.

## 6. Фильтр Блума: «видели ли мы это?» в малой памяти

In [ ]:
#@title Калькулятор фильтра Блума
items = 1000000  #@param {type:"integer"}
false_positive = 0.01  #@param {type:"number"}
m = -items * np.log(false_positive) / np.log(2) ** 2
print(f'{items:,} элементов при доле ложных «видели» {false_positive:.2%}: {m / 8 / 1e6:.1f} МБ '
      f'({m / items:.1f} бит на элемент, {round(m / items * np.log(2))} хеш-функций)')

## 7. Задание 3. Квантизация: где точность падает
Обучаем небольшую нейросеть (9 610 весов) и округляем её веса до 8, 4, 3, 2 бит. Режим **по нейрону** даёт
каждому нейрону свой масштаб — так делают на практике.

In [ ]:
#@title Сжимаем веса и меряем точность
per_neuron = False  #@param {type:"boolean"}
from sklearn.neural_network import MLPClassifier
Xq, yq = load_digits(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(Xq / 16, yq, test_size=0.4, random_state=0, stratify=yq)
mlp = MLPClassifier(hidden_layer_sizes=(128,), max_iter=600, random_state=0).fit(Xtr, ytr)
W0 = [w.copy() for w in mlp.coefs_]
n_w = sum(w.size for w in W0) + sum(b.size for b in mlp.intercepts_)

def quant(w, bits):
    q = 2 ** (bits - 1) - 1
    s = (np.abs(w).max(axis=0, keepdims=True) if per_neuron else np.abs(w).max()) / q
    s = np.where(s == 0, 1, s)
    return np.round(w / s).clip(-q, q) * s

print(f'Весов: {n_w}. Масштаб: {"свой у каждого нейрона" if per_neuron else "один на слой"}\n')
print(f'{"формат":<7}{"размер, КБ":>11}{"точность":>10}')
for name, bits in [('FP32', 32), ('FP16', 16), ('INT8', 8), ('INT4', 4), ('INT3', 3), ('INT2', 2)]:
    if bits == 32:
        mlp.coefs_ = [w.copy() for w in W0]
    elif bits == 16:
        mlp.coefs_ = [w.astype(np.float16).astype(np.float64) for w in W0]
    else:
        mlp.coefs_ = [quant(w, bits) for w in W0]
    print(f'{name:<7}{n_w * bits / 8 / 1024:>11.1f}{(mlp.predict(Xte) == yte).mean():>10.1%}')
mlp.coefs_ = W0

**Что заметить.** До INT4 точность не меняется, на INT3 немного проседает, на INT2 модель разваливается.
Включите **по нейрону** — что стало с INT2? Вывод модуля: сжатие — по замеру на своей проверочной выборке,
а не «по рекомендации из статьи».

---
*© 2026 Наталия Тылевич. Авторский учебный материал курса «Прикладное ML: от данных к системе». Можно выполнять и хранить личную копию для заданий курса; распространять и использовать в других курсах — только с согласия автора. Условия — `LICENSE.md` в репозитории.*